# 03 Preprocessing dan Feature Engineering

Notebook ini digunakan untuk menyiapkan dataset transaksi e-commerce Indonesia sebelum masuk ke tahap modeling.

Tahapan utama pada notebook ini meliputi:

1. Membaca dataset gabungan hasil Notebook 1.
2. Melakukan seleksi data transaksi.
3. Membersihkan kolom numerik dan kategorikal.
4. Membuat fitur baru yang relevan untuk prediksi tingkat permintaan.
5. Membentuk label target `demand_level` berdasarkan `net_qty`.
6. Menyimpan dataset hasil preprocessing untuk digunakan pada Notebook 4.

## Import Library

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Inisialisasi Direktori Project

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATASET_DIR = BASE_DIR / "datasets"
COMBINED_DIR = DATASET_DIR / "combined"
PROCESSED_DIR = DATASET_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

combined_csv_path = COMBINED_DIR / "indonesia_ecommerce_sales_2024_2025_combined.csv"
processed_csv_path = PROCESSED_DIR / "ecommerce_demand_processed.csv"

print("Base directory      :", BASE_DIR)
print("Input dataset path  :", combined_csv_path)
print("Output dataset path :", processed_csv_path)

Base directory      : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code
Input dataset path  : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\combined\indonesia_ecommerce_sales_2024_2025_combined.csv
Output dataset path : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\processed\ecommerce_demand_processed.csv


## Membaca Dataset Gabungan

In [3]:
df = pd.read_csv(combined_csv_path)

print("Jumlah baris awal:", df.shape[0])
print("Jumlah kolom awal:", df.shape[1])

df.head()

Jumlah baris awal: 20404
Jumlah kolom awal: 27


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file,source_year,source_month,source_month_name,source_period,net_qty,is_cancelled,order_year,order_month
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01,2,0,2024,1
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01,2,0,2024,1
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01,1,0,2024,1
3,ORD_0006559,1,700,0,0,Rak / Rak Serbaguna,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT,0,8000,37400,8000,2024-01-01 00:12,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01,1,0,2024,1
4,ORD_0006560,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT,0,8000,21800,8000,2024-01-01 00:36,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01,1,0,2024,1


## Mengecek Kolom Dataset

In [4]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

1. order_id
2. total_qty
3. total_weight_gr
4. total_returned_qty
5. Total Diskon
6. product_categories
7. num_product_categories
8. Status Pesanan
9. Alasan Pembatalan
10. Opsi Pengiriman
11. Metode Pembayaran
12. Kota/Kabupaten
13. Provinsi
14. Ongkos Kirim Dibayar oleh Pembeli
15. Estimasi Potongan Biaya Pengiriman
16. Total Pembayaran
17. Perkiraan Ongkos Kirim
18. Waktu Pesanan Dibuat
19. source_file
20. source_year
21. source_month
22. source_month_name
23. source_period
24. net_qty
25. is_cancelled
26. order_year
27. order_month


## Mengecek Status Pesanan

In [5]:
status_order = (
    df["Status Pesanan"]
    .value_counts(dropna=False)
    .reset_index()
)

status_order.columns = ["Status Pesanan", "total"]

status_order

,Status Pesanan,total
0,Selesai,17416
1,Batal,2738
2,Sedang Dikirim,59
3,"Pesanan diterima, namun Pembeli masih dapat me...",38
4,"Pesanan diterima, namun Pembeli masih dapat me...",34
5,"Pesanan diterima, namun Pembeli masih dapat me...",34
6,Telah Dikirim,30
7,"Pesanan diterima, namun Pembeli masih dapat me...",25
8,"Pesanan diterima, namun Pembeli masih dapat me...",22
9,"Pesanan diterima, namun Pembeli masih dapat me...",6


## Filter Transaksi Selesai

In [6]:
df_filtered = df.copy()

if "Status Pesanan" in df_filtered.columns:
    df_filtered = df_filtered[
        df_filtered["Status Pesanan"].astype(str).str.lower().str.contains("selesai")
    ].copy()

print("Jumlah baris sebelum filter:", df.shape[0])
print("Jumlah baris sesudah filter:", df_filtered.shape[0])
print("Jumlah data yang dibuang   :", df.shape[0] - df_filtered.shape[0])

Jumlah baris sebelum filter: 20404
Jumlah baris sesudah filter: 17416
Jumlah data yang dibuang   : 2988


## Fungsi Konversi Kolom Numerik

In [7]:
def convert_to_numeric(series):
    """
    Mengubah kolom menjadi numerik.
    Fungsi ini dibuat untuk mengantisipasi data angka yang terbaca sebagai teks.
    """
    
    if series.dtype == "object":
        series = (
            series.astype(str)
            .str.replace("Rp", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace("-", "0", regex=False)
            .replace("nan", np.nan)
        )
    
    return pd.to_numeric(series, errors="coerce")

## Membersihkan Kolom Numerik

In [8]:
numeric_columns = [
    "total_qty",
    "total_weight_gr",
    "total_returned_qty",
    "Total Diskon",
    "num_product_categories",
    "Ongkos Kirim Dibayar oleh Pembeli",
    "Estimasi Potongan Biaya Pengiriman",
    "Total Pembayaran",
    "Perkiraan Ongkos Kirim",
    "net_qty"
]

numeric_columns = [
    col for col in numeric_columns 
    if col in df_filtered.columns
]

for col in numeric_columns:
    df_filtered[col] = convert_to_numeric(df_filtered[col])

df_filtered[numeric_columns].info()

<class 'pandas.core.frame.DataFrame'>
Index: 17416 entries, 0 to 20274
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   total_qty                           17416 non-null  int64
 1   total_weight_gr                     17416 non-null  int64
 2   total_returned_qty                  17416 non-null  int64
 3   Total Diskon                        17416 non-null  int64
 4   num_product_categories              17416 non-null  int64
 5   Ongkos Kirim Dibayar oleh Pembeli   17416 non-null  int64
 6   Estimasi Potongan Biaya Pengiriman  17416 non-null  int64
 7   Total Pembayaran                    17416 non-null  int64
 8   Perkiraan Ongkos Kirim              17416 non-null  int64
 9   net_qty                             17416 non-null  int64
dtypes: int64(10)
memory usage: 1.5 MB


## Mengisi Missing Value Kolom Numerik

In [9]:
zero_fill_columns = [
    "total_returned_qty",
    "Total Diskon",
    "Ongkos Kirim Dibayar oleh Pembeli",
    "Estimasi Potongan Biaya Pengiriman",
    "Perkiraan Ongkos Kirim"
]

for col in zero_fill_columns:
    if col in df_filtered.columns:
        df_filtered[col] = df_filtered[col].fillna(0)

median_fill_columns = [
    "total_qty",
    "total_weight_gr",
    "num_product_categories",
    "Total Pembayaran",
    "net_qty"
]

for col in median_fill_columns:
    if col in df_filtered.columns:
        median_value = df_filtered[col].median()
        df_filtered[col] = df_filtered[col].fillna(median_value)

df_filtered[numeric_columns].isnull().sum()

total_qty                             0
total_weight_gr                       0
total_returned_qty                    0
Total Diskon                          0
num_product_categories                0
Ongkos Kirim Dibayar oleh Pembeli     0
Estimasi Potongan Biaya Pengiriman    0
Total Pembayaran                      0
Perkiraan Ongkos Kirim                0
net_qty                               0
dtype: int64

## Membersihkan Kolom Kategorikal

In [10]:
categorical_columns = [
    "product_categories",
    "Opsi Pengiriman",
    "Metode Pembayaran",
    "Kota/Kabupaten",
    "Provinsi"
]

categorical_columns = [
    col for col in categorical_columns 
    if col in df_filtered.columns
]

for col in categorical_columns:
    df_filtered[col] = (
        df_filtered[col]
        .astype(str)
        .str.strip()
        .replace("nan", "Unknown")
        .replace("", "Unknown")
    )

df_filtered[categorical_columns].head()

,product_categories,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi
0,Aksesoris ID Card,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT
1,Celengan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT
2,Plastik / Wadah Plastik,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT
3,Rak / Rak Serbaguna,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT
4,Celengan,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT


## Mengecek Missing Value Setelah Cleaning

In [11]:
missing_after_cleaning = df_filtered.isnull().sum().reset_index()
missing_after_cleaning.columns = ["column", "missing_count"]
missing_after_cleaning = missing_after_cleaning.sort_values("missing_count", ascending=False)

missing_after_cleaning.head(20)

,column,missing_count
8,Alasan Pembatalan,17416
17,Waktu Pesanan Dibuat,1723
0,order_id,0
14,Estimasi Potongan Biaya Pengiriman,0
25,order_year,0
24,is_cancelled,0
23,net_qty,0
22,source_period,0
21,source_month_name,0
20,source_month,0


## Membuat Ulang Kolom net_qty

In [12]:
if "total_qty" in df_filtered.columns and "total_returned_qty" in df_filtered.columns:
    df_filtered["net_qty"] = df_filtered["total_qty"] - df_filtered["total_returned_qty"]
else:
    df_filtered["net_qty"] = df_filtered["total_qty"]

df_filtered["net_qty"] = df_filtered["net_qty"].clip(lower=0)

df_filtered[["total_qty", "total_returned_qty", "net_qty"]].head()

,total_qty,total_returned_qty,net_qty
0,2,0,2
1,2,0,2
2,1,0,1
3,1,0,1
4,1,0,1


## Membuat Fitur Waktu

In [13]:
df_filtered["order_year"] = df_filtered["source_year"]
df_filtered["order_month"] = df_filtered["source_month"]

df_filtered[["source_period", "order_year", "order_month"]].head()

,source_period,order_year,order_month
0,2024-01,2024,1
1,2024-01,2024,1
2,2024-01,2024,1
3,2024-01,2024,1
4,2024-01,2024,1


## Membuat Fitur Tambahan

In [14]:
df_filtered["discount_ratio"] = np.where(
    df_filtered["Total Pembayaran"] > 0,
    df_filtered["Total Diskon"] / df_filtered["Total Pembayaran"],
    0
)

df_filtered["shipping_ratio"] = np.where(
    df_filtered["Total Pembayaran"] > 0,
    df_filtered["Perkiraan Ongkos Kirim"] / df_filtered["Total Pembayaran"],
    0
)

df_filtered["is_cod"] = np.where(
    df_filtered["Metode Pembayaran"].astype(str).str.lower().str.contains("cod"),
    1,
    0
)

df_filtered["has_shipping_discount"] = np.where(
    df_filtered["Estimasi Potongan Biaya Pengiriman"] > 0,
    1,
    0
)

df_filtered[[
    "discount_ratio",
    "shipping_ratio",
    "is_cod",
    "has_shipping_discount"
]].head()

,discount_ratio,shipping_ratio,is_cod,has_shipping_discount
0,0.0,0.800000,0,1
1,0.0,0.280403,1,1
2,0.0,0.431276,1,1
3,0.0,0.213904,1,1
4,0.0,0.366972,0,1


## Membatasi Nilai Rasio

In [15]:
ratio_columns = ["discount_ratio", "shipping_ratio"]

for col in ratio_columns:
    df_filtered[col] = df_filtered[col].replace([np.inf, -np.inf], 0)
    df_filtered[col] = df_filtered[col].fillna(0)
    df_filtered[col] = df_filtered[col].clip(lower=0)

df_filtered[ratio_columns].describe()

,discount_ratio,shipping_ratio
count,17416.000000,17416.000000
mean,0.000675,0.524349
std,0.012026,0.949168
min,0.000000,0.000000
25%,0.000000,0.264085
50%,0.000000,0.421053
75%,0.000000,0.650518
max,0.421053,87.272727


## Mengecek Distribusi net_qty

In [16]:
df_filtered["net_qty"].describe()

count    17416.000000
mean         2.468190
std          7.258661
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        256.000000
Name: net_qty, dtype: float64

In [17]:
df_filtered["net_qty"].value_counts().sort_index().head(20)

net_qty
0        70
1     10218
2      3897
3      1130
4       654
5       428
6       293
7        95
8        81
9        31
10      162
11       27
12       45
13       13
14        8
15       29
16        7
17        4
18        4
19        2
Name: count, dtype: int64

## Membuat Label Target demand_level

In [18]:
def create_demand_label(net_qty):
    if net_qty <= 1:
        return "Low Demand"
    elif net_qty <= 3:
        return "Medium Demand"
    else:
        return "High Demand"


df_filtered["demand_level"] = df_filtered["net_qty"].apply(create_demand_label)

df_filtered["demand_level"].value_counts()

demand_level
Low Demand       10288
Medium Demand     5027
High Demand       2101
Name: count, dtype: int64

## Mengecek Distribusi Label Target

In [19]:
demand_distribution = (
    df_filtered["demand_level"]
    .value_counts()
    .reset_index()
)

demand_distribution.columns = ["demand_level", "total"]

demand_distribution

,demand_level,total
0,Low Demand,10288
1,Medium Demand,5027
2,High Demand,2101


## Encoding Label Target

In [20]:
demand_label_map = {
    "Low Demand": 0,
    "Medium Demand": 1,
    "High Demand": 2
}

df_filtered["label"] = df_filtered["demand_level"].map(demand_label_map)

df_filtered[["demand_level", "label"]].head()

,demand_level,label
0,Medium Demand,1
1,Medium Demand,1
2,Low Demand,0
3,Low Demand,0
4,Low Demand,0


## Memilih Fitur untuk Modeling

In [21]:
selected_columns = [
    "total_weight_gr",
    "Total Diskon",
    "num_product_categories",
    "Ongkos Kirim Dibayar oleh Pembeli",
    "Estimasi Potongan Biaya Pengiriman",
    "Total Pembayaran",
    "Perkiraan Ongkos Kirim",
    "discount_ratio",
    "shipping_ratio",
    "is_cod",
    "has_shipping_discount",
    "order_year",
    "order_month",
    "product_categories",
    "Opsi Pengiriman",
    "Metode Pembayaran",
    "Kota/Kabupaten",
    "Provinsi",
    "net_qty",
    "demand_level",
    "label"
]

selected_columns = [
    col for col in selected_columns 
    if col in df_filtered.columns
]

df_processed = df_filtered[selected_columns].copy()

print("Jumlah baris dataset processed:", df_processed.shape[0])
print("Jumlah kolom dataset processed:", df_processed.shape[1])

df_processed.head()

Jumlah baris dataset processed: 17416
Jumlah kolom dataset processed: 21


,total_weight_gr,Total Diskon,num_product_categories,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,discount_ratio,shipping_ratio,is_cod,has_shipping_discount,order_year,order_month,product_categories,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,net_qty,demand_level,label
0,50,0,1,0,8000,10000,8000,0.0,0.800000,0,1,2024,1,Aksesoris ID Card,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,2,Medium Demand,1
1,1000,0,1,0,10000,35663,10000,0.0,0.280403,1,1,2024,1,Celengan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,2,Medium Demand,1
2,600,0,1,0,10000,23187,10000,0.0,0.431276,1,1,2024,1,Plastik / Wadah Plastik,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,1,Low Demand,0
3,700,0,1,0,8000,37400,8000,0.0,0.213904,1,1,2024,1,Rak / Rak Serbaguna,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT,1,Low Demand,0
4,500,0,1,0,8000,21800,8000,0.0,0.366972,0,1,2024,1,Celengan,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT,1,Low Demand,0


## Mengecek Missing Value Dataset Processed

In [22]:
processed_missing = df_processed.isnull().sum().reset_index()
processed_missing.columns = ["column", "missing_count"]
processed_missing = processed_missing.sort_values("missing_count", ascending=False)

processed_missing

,column,missing_count
0,total_weight_gr,0
11,order_year,0
19,demand_level,0
18,net_qty,0
17,Provinsi,0
16,Kota/Kabupaten,0
15,Metode Pembayaran,0
14,Opsi Pengiriman,0
13,product_categories,0
12,order_month,0


## Mengecek Tipe Data Dataset Processed

In [23]:
df_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17416 entries, 0 to 20274
Data columns (total 21 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   total_weight_gr                     17416 non-null  int64  
 1   Total Diskon                        17416 non-null  int64  
 2   num_product_categories              17416 non-null  int64  
 3   Ongkos Kirim Dibayar oleh Pembeli   17416 non-null  int64  
 4   Estimasi Potongan Biaya Pengiriman  17416 non-null  int64  
 5   Total Pembayaran                    17416 non-null  int64  
 6   Perkiraan Ongkos Kirim              17416 non-null  int64  
 7   discount_ratio                      17416 non-null  float64
 8   shipping_ratio                      17416 non-null  float64
 9   is_cod                              17416 non-null  int32  
 10  has_shipping_discount               17416 non-null  int32  
 11  order_year                          17416 non-

## Menyimpan Dataset Processed

In [24]:
df_processed.to_csv(processed_csv_path, index=False, encoding="utf-8-sig")

print("Dataset hasil preprocessing berhasil disimpan ke:")
print(processed_csv_path)

Dataset hasil preprocessing berhasil disimpan ke:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\processed\ecommerce_demand_processed.csv


## Validasi Dataset dengan PySpark

In [25]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Preprocessing E-Commerce Demand Dataset")
    .getOrCreate()
)

spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(processed_csv_path))
)

print("Jumlah baris Spark DataFrame:", spark_df.count())
print("Jumlah kolom Spark DataFrame:", len(spark_df.columns))

spark_df.printSchema()

Jumlah baris Spark DataFrame: 17416
Jumlah kolom Spark DataFrame: 21
root
 |-- total_weight_gr: integer (nullable = true)
 |-- Total Diskon: integer (nullable = true)
 |-- num_product_categories: integer (nullable = true)
 |-- Ongkos Kirim Dibayar oleh Pembeli: integer (nullable = true)
 |-- Estimasi Potongan Biaya Pengiriman: integer (nullable = true)
 |-- Total Pembayaran: integer (nullable = true)
 |-- Perkiraan Ongkos Kirim: integer (nullable = true)
 |-- discount_ratio: double (nullable = true)
 |-- shipping_ratio: double (nullable = true)
 |-- is_cod: integer (nullable = true)
 |-- has_shipping_discount: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- product_categories: string (nullable = true)
 |-- Opsi Pengiriman: string (nullable = true)
 |-- Metode Pembayaran: string (nullable = true)
 |-- Kota/Kabupaten: string (nullable = true)
 |-- Provinsi: string (nullable = true)
 |-- net_qty: integer (nullable = tru

## Cek Distribusi Target dengan PySpark

In [26]:
from pyspark.sql.functions import col

(
    spark_df
    .groupBy("demand_level", "label")
    .count()
    .orderBy("label")
    .show(truncate=False)
)

+-------------+-----+-----+
|demand_level |label|count|
+-------------+-----+-----+
|Low Demand   |0    |10288|
|Medium Demand|1    |5027 |
|High Demand  |2    |2101 |
+-------------+-----+-----+

